In [1]:
%pip install google-cloud-bigquery
%pip install azure-storage-blob
%pip install db-dtypes

%pip install google-cloud-bigquery-storage
%pip install pandas-gbq
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# =========================
# Inicialização Databricks
# =========================

try:
    # Cria widgets no Databricks
    dbutils.library.restartPython()
except NameError:
    # Fallback para execução fora do Databricks (ex: testes locais em Jupyter/VSCode)
    print("⚠️ dbutils não encontrado, usando valores locais para teste.")

⚠️ dbutils não encontrado, usando valores locais para teste.


In [3]:
# =========================
# Imports
# =========================

from pathlib import Path
import os
import json
import tempfile
import datetime as dt

import db_dtypes

# =========================
# 1. Definir raiz do projeto
# =========================

BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists():

    if BASE_DIR.parent == BASE_DIR:
        raise FileNotFoundError(
            "❌ Pasta 'src' não encontrada em nenhum nível acima."
        )

    BASE_DIR = BASE_DIR.parent

print(f"✅ BASE_DIR localizado: {BASE_DIR}")

# =========================
# 2. Configuração das credenciais GCP
# =========================

cred_path = None

# Variáveis que serão usadas nas demais células
AZURE_STORAGE_ACCOUNT = None
AZURE_STORAGE_KEY = None

try:

    # ==========================================
    # Execução Databricks + Azure Key Vault
    # ==========================================

    print("✅ Ambiente Databricks detectado.")

    # --------------------------------------------------
    # Credenciais Google BigQuery
    # --------------------------------------------------

    secret_json = dbutils.secrets.get(
        scope="kvfiaptechprod",
        key="GOOGLE-APPLICATION-CREDENTIALS-JSON"
    )

    temp_file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".json",
        delete=False
    )

    temp_file.write(secret_json)
    temp_file.close()

    cred_path = temp_file.name

    print(
        "✅ Credenciais BigQuery carregadas do Azure Key Vault."
    )

    # --------------------------------------------------
    # Credenciais Azure Storage
    # --------------------------------------------------

    AZURE_STORAGE_ACCOUNT = dbutils.secrets.get(
        scope="kvfiaptechprod",
        key="AZURE-STORAGE-ACCOUNT"
    )

    AZURE_STORAGE_KEY = dbutils.secrets.get(
        scope="kvfiaptechprod",
        key="AZURE-STORAGE-KEY"
    )

    print(
        f"✅ Storage Account carregado: {AZURE_STORAGE_ACCOUNT}"
    )

except NameError:

    # ==========================================
    # Execução Local (VSCode / Jupyter)
    # ==========================================

    print(
        "⚠️ dbutils não encontrado. "
        "Utilizando credenciais locais."
    )

    # --------------------------------------------------
    # Credenciais BigQuery locais
    # --------------------------------------------------

    cred_file = (
        "tough-medley-505300-k1-164371097431.json"
    )

    cred_path = (
        BASE_DIR
        / "credenciais"
        / cred_file
    )

    if not cred_path.exists():

        raise FileNotFoundError(
            f"❌ Arquivo não encontrado: {cred_path}"
        )

    cred_path = str(cred_path)

    print(
        f"✅ Arquivo de credenciais localizado em: "
        f"{cred_path}"
    )

    # --------------------------------------------------
    # Credenciais Azure Storage locais
    # --------------------------------------------------

    AZURE_STORAGE_ACCOUNT = os.getenv(
        "AZURE_STORAGE_ACCOUNT"
    )

    AZURE_STORAGE_KEY = os.getenv(
        "AZURE_STORAGE_KEY"
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Erro ao carregar credenciais: {e}"
    )

# =========================
# 3. Variáveis de ambiente
# =========================

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(
    cred_path
)

if AZURE_STORAGE_ACCOUNT:
    os.environ["AZURE_STORAGE_ACCOUNT"] = (
        AZURE_STORAGE_ACCOUNT
    )

if AZURE_STORAGE_KEY:
    os.environ["AZURE_STORAGE_KEY"] = (
        AZURE_STORAGE_KEY
    )

print(
    "✅ Variáveis de ambiente configuradas."
)

# =========================
# 4. Validação das credenciais
# =========================

if os.path.exists(str(cred_path)):

    print(
        f"✅ Arquivo de credenciais disponível em: "
        f"{cred_path}"
    )

else:

    raise FileNotFoundError(
        f"❌ Arquivo de credenciais não encontrado: "
        f"{cred_path}"
    )

print(
    f"✅ Storage Account: {AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Storage Key carregada: "
    f"{bool(AZURE_STORAGE_KEY)}"
)

✅ BASE_DIR localizado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23
✅ Ambiente Databricks detectado.
⚠️ dbutils não encontrado. Utilizando credenciais locais.
✅ Arquivo de credenciais localizado em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json
✅ Variáveis de ambiente configuradas.
✅ Arquivo de credenciais disponível em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json
✅ Storage Account: stfiapoin4ci2kb4w7c
✅ Storage Key carregada: True


In [4]:
# =========================
# Parâmetros vindos do Databricks
# =========================

try:

    # ------------------------------------------
    # Widgets para execução via Databricks Job
    # ou Azure Data Factory
    # ------------------------------------------

    dbutils.widgets.text(
        "BRONZE_CONTAINER",
        "bronze"
    )

    dbutils.widgets.text(
        "TABLES",
        ""
    )

    # ------------------------------------------
    # Leitura dos parâmetros
    # ------------------------------------------

    BRONZE_CONTAINER = (
        dbutils.widgets.get("BRONZE_CONTAINER")
        or "bronze"
    )

    TABLES = (
        [
            table.strip()
            for table in dbutils.widgets.get(
                "TABLES"
            ).split(",")
            if table.strip()
        ]
        if dbutils.widgets.get("TABLES")
        else []
    )

    print(
        "✅ Widgets Databricks carregados com sucesso."
    )

except NameError:

    # ------------------------------------------
    # Execução local
    # VSCode / Jupyter Notebook
    # ------------------------------------------

    print(
        "⚠️ dbutils não encontrado, "
        "utilizando configuração local."
    )

    BRONZE_CONTAINER = "bronze"

    TABLES = []

# =========================
# Exibição dos parâmetros
# =========================

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Quantidade de tabelas recebidas: "
    f"{len(TABLES)}"
)

if TABLES:

    print(
        "✅ Tabelas configuradas:"
    )

    for table in TABLES:

        print(
            f"   • {table}"
        )

else:

    print(
        "⚠️ Nenhuma tabela foi informada via widget."
    )

# =========================
# Variáveis disponíveis para
# as próximas células
# =========================

print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Storage Key carregada: "
    f"{bool(AZURE_STORAGE_KEY)}"
)

⚠️ dbutils não encontrado, utilizando configuração local.
✅ Container Bronze: bronze
✅ Quantidade de tabelas recebidas: 0
⚠️ Nenhuma tabela foi informada via widget.
✅ Storage Account: stfiapoin4ci2kb4w7c
✅ Storage Key carregada: True


In [5]:
# =========================
# Configuração dos clientes
# =========================

from google.cloud import bigquery
from azure.storage.blob import BlobServiceClient

# =========================
# Cliente BigQuery
# =========================

try:

    if not cred_path:

        raise ValueError(
            "❌ Credenciais GCP não configuradas."
        )

    print(
        f"✅ Credenciais encontradas em: "
        f"{cred_path}"
    )

    # ------------------------------------------
    # Projeto Google Cloud
    # ------------------------------------------

    project_id = os.getenv(
        "GCP_PROJECT_ID",
        "tough-medley-505300-k1"
    )

    # ------------------------------------------
    # Inicializa cliente BigQuery
    # ------------------------------------------

    bq_client = bigquery.Client.from_service_account_json(
        cred_path,
        project=project_id
    )

    print(
        "✅ Cliente BigQuery inicializado com sucesso."
    )

except Exception as e:

    print(
        f"❌ Erro ao inicializar cliente BigQuery: {e}"
    )

    bq_client = None

# =========================
# Cliente Azure Blob
# =========================

try:

    # ------------------------------------------
    # Validação das credenciais Azure
    # ------------------------------------------

    if not AZURE_STORAGE_ACCOUNT:

        raise ValueError(
            "❌ AZURE_STORAGE_ACCOUNT não configurado."
        )

    if not AZURE_STORAGE_KEY:

        raise ValueError(
            "❌ AZURE_STORAGE_KEY não configurado."
        )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        "✅ Storage Key carregada com sucesso."
    )

    # ------------------------------------------
    # Inicializa Blob Storage
    # ------------------------------------------

    blob_service_client = BlobServiceClient(
        account_url=f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net",
        credential=AZURE_STORAGE_KEY
    )

    print(
        "✅ Cliente Azure Blob inicializado com sucesso."
    )

    # ------------------------------------------
    # Teste rápido da conexão
    # ------------------------------------------

    container_client = (
        blob_service_client.get_container_client(
            BRONZE_CONTAINER
        )
    )

    print(
        f"✅ Acesso validado ao container "
        f"'{BRONZE_CONTAINER}'."
    )

except Exception as e:

    print(
        f"❌ Erro ao inicializar cliente Azure Blob: {e}"
    )

    blob_service_client = None

# =========================
# Resumo de inicialização
# =========================

print(
    f"✅ BigQuery disponível: {bq_client is not None}"
)

print(
    f"✅ Azure Blob disponível: "
    f"{blob_service_client is not None}"
)

✅ Credenciais encontradas em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json
✅ Cliente BigQuery inicializado com sucesso.
✅ Storage Account: stfiapoin4ci2kb4w7c
✅ Storage Key carregada com sucesso.
✅ Cliente Azure Blob inicializado com sucesso.
✅ Acesso validado ao container 'bronze'.
✅ BigQuery disponível: True
✅ Azure Blob disponível: True


In [6]:
# =========================
# Função de exportação
# =========================

def export_bigquery_table_to_blob(
    source_table: str,
    blob_container: str,
    blob_name: str
):
    """
    Exporta uma tabela do BigQuery
    para Azure Blob Storage em Parquet.
    """

    try:

        # ------------------------------------------
        # Validação dos clientes
        # ------------------------------------------

        if bq_client is None:

            raise ValueError(
                "❌ Cliente BigQuery não inicializado."
            )

        if blob_service_client is None:

            raise ValueError(
                "❌ Cliente Azure Blob não inicializado."
            )

        # ------------------------------------------
        # Consulta BigQuery
        # ------------------------------------------

        query = f"""
        SELECT *
        FROM `{source_table}`
        """

        print(
            f"✅ Executando consulta: {source_table}"
        )

        query_job = bq_client.query(
            query,
            location="US"
        )

        # ------------------------------------------
        # Converte para DataFrame
        # ------------------------------------------

        df = query_job.to_dataframe()

        print(
            f"✅ Registros retornados: {len(df):,}"
        )

        # ------------------------------------------
        # Colunas de auditoria
        # ------------------------------------------

        df["_ingested_at"] = (
            dt.datetime.now(
                dt.timezone.utc
            ).isoformat()
        )

        df["_source_table"] = source_table

        # ------------------------------------------
        # Diretório temporário
        # ------------------------------------------

        temp_dir = BASE_DIR / "tmp"

        temp_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        parquet_file = (
            temp_dir / blob_name
        )

        # ------------------------------------------
        # Salva Parquet
        # ------------------------------------------

        df.to_parquet(
            parquet_file,
            index=False
        )

        print(
            f"✅ Arquivo parquet criado: "
            f"{parquet_file}"
        )

        # ------------------------------------------
        # Upload Blob Storage
        # ------------------------------------------

        blob_client = (
            blob_service_client.get_blob_client(
                container=blob_container,
                blob=blob_name
            )
        )

        with open(parquet_file, "rb") as data:

            blob_client.upload_blob(
                data,
                overwrite=True
            )

        print(
            f"✅ Exportado {source_table} "
            f"para azure://{blob_container}/{blob_name}"
        )

    except Exception as e:

        print(
            f"❌ Erro ao exportar "
            f"{source_table}: {e}"
        )

        raise

In [ ]:
# =========================
# Ingestão Batch
# =========================

import datetime as dt

# ------------------------------------------
# Garantia de existência da variável TABLES
# ------------------------------------------

try:
    TABLES
except NameError:
    TABLES = []

# ------------------------------------------
# Data de execução
# ------------------------------------------

execution_date = dt.datetime.now()

date_suffix = execution_date.strftime("%Y-%m-%d")
print(f"✅ Data de execução: {date_suffix}")

# ------------------------------------------
# Configuração padrão
# ------------------------------------------

tables_to_process = [
    {
        "source_table": (
            "basedosdados.br_inep_avaliacao_alfabetizacao.uf"
        ),
        "blob_name": (
            f"{date_suffix}_uf.parquet"
        )
    }
]

# ------------------------------------------
# Caso venham tabelas do widget
# ------------------------------------------

if TABLES:
    tables_to_process = [
        {
            "source_table": table,
            "blob_name": (
                f"{date_suffix}_"
                f"{table.split('.')[-1]}.parquet"
            )
        }

        for table in TABLES
    ]

# ------------------------------------------
# Processamento
# ------------------------------------------

for table_config in tables_to_process:
    source_table = table_config["source_table"]
    blob_name = table_config["blob_name"]
    print(
        f"✅ Processando tabela: "
        f"{source_table}"
    )

    export_bigquery_table_to_blob(
        source_table=source_table,
        blob_container=BRONZE_CONTAINER,
        blob_name=blob_name
    )

print(
    "✅ Ingestão finalizada com sucesso."
)

✅ Data de execução: 2026-08-23
✅ Processando tabela: basedosdados.br_inep_avaliacao_alfabetizacao.uf
✅ Executando consulta: basedosdados.br_inep_avaliacao_alfabetizacao.uf
✅ Registros retornados: 145
✅ Arquivo parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/tmp/2026-08-23_uf.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.uf para azure://bronze/2026-08-23_uf.parquet
✅ Ingestão finalizada com sucesso.


In [8]:
import datetime as dt

# Lista de tabelas
TABLES = [
    "basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio"
    #,
   # "basedosdados.br_inep_avaliacao_alfabetizacao.municipio",
   # "basedosdados.br_inep_avaliacao_alfabetizacao.alunos"
]

# Data atual para sufixo
date_suffix = dt.datetime.now().strftime("%Y-%m-%d")
container = os.getenv("AZURE_CONTAINER_BRONZE", "bronze")

# Loop sobre todas as tabelas
for table in TABLES:
# Usa apenas o último pedaço do nome da tabela para o arquivo
    table_suffix = table.split(".")[-1]
    blob_name_with_date = f"{date_suffix}_{table_suffix}.parquet"

    export_bigquery_table_to_blob(
        source_table=table,
        blob_container=container,
        blob_name=blob_name_with_date
    )


✅ Executando consulta: basedosdados.br_inep_avaliacao_alfabetizacao.uf


✅ Registros retornados: 145
✅ Arquivo parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/tmp/2026-08-23_uf.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.uf para azure://bronze/2026-08-23_uf.parquet
✅ Executando consulta: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
✅ Registros retornados: 3
✅ Arquivo parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/tmp/2026-08-23_meta_alfabetizacao_brasil.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil para azure://bronze/2026-08-23_meta_alfabetizacao_brasil.parquet
✅ Executando consulta: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
✅ Registros retornados: 81
✅ Arquivo parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/tmp/2026-08-23_meta_alfabetizacao_uf.parquet
✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf para azure://bronze/2026-08-